In [ ]:
# Create engine for sqlalchemy

from sqlalchemy import ForeignKey
from sqlalchemy.orm import relationship
from sqlalchemy.orm import DeclarativeBase
from sqlalchemy import Table, Column, Integer, String, MetaData
from importlib import metadata


from sqlalchemy import create_engine

engine = create_engine("sqlite:///example.db", echo=False)

# Create table


class Base(DeclarativeBase):
    pass


class Category(Base):
    __tablename__ = "categories"

    id = Column(Integer, primary_key=True)
    name = Column(String, nullable=True)
    description = Column(String, nullable=True)
    products = relationship("Product", back_populates="category")

    def __repr__(self):
        return f"Category(id={self.id!r}, name={self.name!r}, description={self.description!r})"


class Product(Base):
    __tablename__  = "products"

    id = Column(Integer, primary_key=True)
    name = Column(String, nullable=True)
    description = Column(String, nullable=True)
    category_id = Column(Integer, ForeignKey("categories.id"))
    category = relationship("Category", back_populates="products")


    def __repr__(self):
        return f"Product(id={self.id!r}, name={self.name!r}, description={self.description!r}, category_id={self.category_id!r})"   
    


Base.metadata.create_all(engine)

In [ ]:
# Populate the database with some data
data = [
    {
        "name": "Category 1",
        "description": "Description 1",
        "products": [
            {"name": "Product 1", "description": "Description 1"},
            {"name": "Product 2", "description": "Description 2"},
        ],
    },
    {
        "name": "Category 2",
        "description": "Description 2",
        "products": [
            {"name": "Product 3", "description": "Description 3"},
            {"name": "Product 4", "description": "Description 4"},
        ],
    },
]

# Create session
from sqlalchemy.orm import Session

session = Session(engine)

for category_data in data:
    product_data = category_data.pop("products")

    category = Category(**category_data)
    session.add(category)
    session.flush()

    for product_dict in product_data:
        product = Product(**product_dict)
        product.category = category
        session.add(product)
    
    session.commit()


In [ ]:
data = [
    {
        "name": "Category 1",
        "description": "Description 1",
        "products": [
            {"name": "Product 1", "description": "Description 1"},
            {"name": "Product 2", "description": "Description 2"},
        ],
    },
    {
        "name": "Category 2",
        "description": "Description 2",
        "products": [
            {"name": "Product 3", "description": "Description 3"},
            {"name": "Product 4", "description": "Description 4"},
        ],
    },
]

session = Session(engine)

categores = []

for category_data in data:
    product_data = category_data["products"]

    category = Category(
        name=category_data["name"],
        description=category_data["description"],
        products=[Product(**product) for product in product_data]
    )

    categores.append(category)

session.add_all(categores)
session.commit()


In [ ]:
for i in session.query(Product).all():
    print(i)

In [ ]:
# Write a method that takes a csv file as input as populates the database
import csv


def populate_categories_from_csv(file_path):
    file = open(file_path, "r")
    reader = csv.reader(file)
    categories = []
    for row in reader:
        categories.append(
            Category(
                name=row[0], 
                description=row[1]
            )
        )
    
    session.add_all(categories)
    session.commit()


def populate_products_from_csv(file_path):
    file = open(file_path, "r")
    reader = csv.reader(file)
    products = []
    for row in reader:
        products.append(
            Product(
                name=row[0],
                description=row[1],
                category_id=row[2]
            )
        )

    session.add_all(products)
    session.commit()


populate_categories_from_csv("MOCK_DATA.csv")
populate_products_from_csv("MOCK_DATA (1).csv")

In [ ]:
session.query(Product).delete()
session.query(Category).delete()
session.commit()

In [ ]:
session = Session(engine)
session.query(Product).filter_by(name="Product 1").update({"name": "Laptop"})

In [ ]:
session.query(Product).filter_by(name="Laptop").all()

In [ ]:
session.query(Product).filter_by(id=1).one_or_none()

In [ ]:
# Annotate the product query by the average of category_id

from sqlalchemy import func

session.query(Category.name, func.count(Product.id).label("product_count")).join(Product).group_by(Category.name).all()

In [ ]:
# Find the names of all Products that belong to a Category specifically named "Electronics".

session.query(Product.name).join(Category).filter(Category.name == "Category 1").all()

In [ ]:
# List all Categories, but only include those that have more than 5 products.

session.query(Category).join(Product).group_by(Category.id).having(func.count(Product.id) > 5).all()

In [ ]:
# Find all Categories that currently have zero products.

session.query(Category).join(Product).group_by(Category.id).having(func.count(Product.id) > 0).all()

In [ ]:
# Find all Categories that currently have zero products.

session.query(Category.name).outerjoin(Product).group_by(Category.id).filter(Product.id == None).all()

In [ ]:
# Find all Categories name with the product count

session.query(Category.name, func.count(Product.id).label("product_count")).join(Product).group_by(Category.name).all()

In [ ]:
# For every Product, show its name and the total number of products that exist in its same category.

from sqlalchemy import select


subquery = (
    select(func.count(Product.id))
    .where(Product.category_id == Category.id)
    .scalar_subquery()
)

for product in session.query(Product.name, subquery.label("product_count")).all():
    print(product)


In [ ]:
# For every Product, show its name and the total number of products that exist in its same category.

from sqlalchemy.orm import aliased

# Create an alias for Product to compare the table with itself
P = aliased(Product, name="P")

# Subquery: Count products in P that have the same category_id as the current Product
subquery = (
    select(func.count(P.id))
    .where(P.category_id == Product.category_id)
    .scalar_subquery()
)

session.query(Product.name, subquery.label("product_count")).all()

In [ ]:
# Find the Category that has the most products. Return a tuple containing the Category.name and the count of its products.
P = aliased(Product, name="P")

session.query(Category.name, func.count(P.id)).join(P).group_by(Category.id).order_by(func.count(P.id).desc()).all()

In [ ]:
# Find the Category that has the most products. Return a tuple containing the Category.name and the count of its products.

P = aliased(Product, name="P")

(
    session.query(Category.name, func.count(P.id))
    .join(P)
    .group_by(Category.id)
    .order_by(func.count(P.id).desc())
    .all()
)

In [ ]:
# Find all Products whose description contains "Smart" (case-insensitive) and category_id > 50

(
    session.query(Product)
    .filter(Product.description.ilike("%smart%"))
    .filter(Product.category_id > 50)
    .all()
)

In [ ]:
# Update the description of all products in Category 73.

(
    session.query(Product)
    .filter(Product.category_id == 73)
    .update({Product.description: "Updated description"}, synchronize_session="fetch")
)
session.commit()

In [ ]:
# Select names of Categories that have at least one "Gardening" product.
from sqlalchemy import exists

(
    session.query(Category.name)
    .filter(exists()
            .where(Product.category_id == Category.id)
            .where(Product.description.ilike("%Gardening%")))
    .all()
)

In [ ]:
# Find the Average Number of Products per Category across the entire database.

product_counts_cte = (
    session.query(
        Product.category_id, 
        func.count(Product.id).label("product_count_per_category")
    )
    .group_by(Product.category_id)
    .cte("product_counts")
)

(
    session.query(
        func.avg(product_counts_cte.c.product_count_per_category).label("avg_products_per_category")
    ).scalar()
)


In [ ]:
# Find all products that belong to Category ID 5 and have an ID greater than 50.

(
    session.query(Product)
    .filter(Product.id > 50, Product.category_id == 5)
    .all()
)

In [ ]:
# Find all categories whose name contains "Food" regardless of capitalization.

(
    session.query(Category)
    .filter(Category.name.ilike("%food%"))
    .all()
)

In [ ]:
# Find products whose description starts specifically with the word "Organic".

(
    session.query(Product)
    .filter(Product.description.startswith("Seeds"))
    .all()
)

In [ ]:
# Find all products that belong to Category 1, 10, or 99.

(
    session.query(Product)
    .filter(Product.category_id.in_([1, 10, 99]))
    .all()
)

In [ ]:
# Find all products that do not have a category assigned.

(
    session.query(Product)
    .filter(Product.category_id == None)
    .all()
)

In [ ]:
# Find products that are either in Category 7 OR have the name "Laptop".
from sqlalchemy import or_

(
    session.query(Product)
    .filter(or_(Product.category_id == 7, Product.name == "Laptop"))
    .all()
)

In [ ]:
# Find products that are either in Category 7 AND have the name "Laptop".
from sqlalchemy import and_

(
    session.query(Product)
    .filter(and_(Product.category_id == 7, Product.name == "Laptop"))
    .all()
)